In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModel, pipeline
import torch
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import torch.nn.functional as F

print("İP 3 Başladı: Samimiyet Skoru (S) Modülü...")

# -------------------------------------------------------------------------
# ADIM 1: GİRDİLERİ HAZIRLA
# -------------------------------------------------------------------------
# 1. Örnek Motivasyon Mektubu (CV'nizdeki AIESEC bölümü)
sample_motivation_letter = """
I would like to do activities and things that makes me more self-confident. I like to travel too. 
I wanted to connect these two things together thats why I decided to become Global Volunteer of AIESEC. 
I was working at school with kids on project Global Classroom in Iasi Romania. 
I lived there for 2 months with people from all over the world.
"""

# 2. İP 2'den gelen projeleri yükle
try:
    df_sonuc = pd.read_pickle("IP2_Ciktisi_M_Skorlu_Projeler.pkl")
    print(f"-> {len(df_sonuc)} adet puanlanmış proje yüklendi.")
except FileNotFoundError:
    print("HATA: 'IP2_Ciktisi_M_Skorlu_Projeler.pkl' bulunamadı.")
    raise

# -------------------------------------------------------------------------
# ADIM 2: HİLE TESPİTİ (AI DETECTION) MODELİ
# -------------------------------------------------------------------------
print("\nAdım 2: Hile Tespiti (AI Detector) modeli yükleniyor...")
# Bu model, metnin yapay zekâ (GPT) tarafından yazılma olasılığını ölçer.
ai_model_name = "roberta-base-openai-detector"
ai_tokenizer = AutoTokenizer.from_pretrained(ai_model_name)
ai_model = AutoModelForSequenceClassification.from_pretrained(ai_model_name)

def detect_ai_generated(text):
    """Metnin AI tarafından yazılma olasılığını (0-1) döndürür."""
    inputs = ai_tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        logits = ai_model(**inputs).logits
    probs = F.softmax(logits, dim=1)
    # 0: Human, 1: AI (Modelin etiket yapısına göre Fake=1)
    ai_probability = probs[0][1].item() 
    return ai_probability

# Örnek mektubun AI skorunu hesapla
ai_score = detect_ai_generated(sample_motivation_letter)
print(f"-> Mektubun Yapay Zekâ Olma İhtimali: %{ai_score * 100:.2f}")
# Samimiyet puanı için tersini alıyoruz (İnsan yazımı = Yüksek Puan)
human_score = (1 - ai_score) * 100

# -------------------------------------------------------------------------
# ADIM 3: BAĞLAMSAL CİDDİYET (CONTEXTUAL SIMILARITY)
# -------------------------------------------------------------------------
print("\nAdım 3: Bağlamsal Benzerlik modeli yükleniyor (XLM-R)...")
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
embed_model_name = "xlm-roberta-base"
embed_tokenizer = AutoTokenizer.from_pretrained(embed_model_name)
embed_model = AutoModel.from_pretrained(embed_model_name).to(device)

def get_embedding(text):
    inputs = embed_tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = embed_model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).cpu().numpy()

# Mektubun vektörünü çıkar
letter_vector = get_embedding(sample_motivation_letter)

def calculate_s_score(row):
    # 1. Proje tanımının vektörünü al (veya yeniden hesapla)
    # İP 1'de 'tam_aciklama' vektörünü kaydetmemiştik, burada hızlıca hesaplayalım
    project_text = row['tam_aciklama']
    project_vector = get_embedding(str(project_text))
    
    # 2. Kosinüs Benzerliği (Mektup vs Proje Tanımı)
    similarity = cosine_similarity(letter_vector, project_vector)[0][0]
    
    # 3. Nihai S Skoru: %50 İnsan Yazımı + %50 Proje Benzerliği
    # (Benzerlik genelde 0-1 arasıdır, 100 ile çarpıyoruz)
    context_score = similarity * 100
    
    # Formül: S = (Human_Score * 0.4) + (Context_Score * 0.6)
    # Bağlam (Context) daha belirleyicidir çünkü her mektup insan yazımı olabilir ama alakasız olabilir.
    final_s = (human_score * 0.4) + (context_score * 0.6)
    
    return final_s, human_score, context_score

# -------------------------------------------------------------------------
# ADIM 4: HESAPLAMA VE SONUÇLAR
# -------------------------------------------------------------------------
print("\nSamimiyet Skorları Hesaplanıyor (İlk 20 proje için)...")

# Performans için sadece en iyi 20 projeyi puanlayalım
df_top20 = df_sonuc.head(20).copy()

# Apply fonksiyonu birden fazla sütun döndürecek
metrics = df_top20.apply(calculate_s_score, axis=1, result_type='expand')
df_top20[['S_Skoru', 'Insan_Puani', 'Baglam_Puani']] = metrics

# Sonuçları Göster
print("\n🏆 EN SAMİMİ EŞLEŞMELER (S-Skoru) 🏆")
print(f"Referans Mektup AI Skoru: {ai_score:.4f} (Düşük olması iyidir)")
print("-" * 80)

for i, row in df_top20.head(5).iterrows():
    proje_isim = row['link'].split('/')[-1].replace('_en', '').upper()
    print(f"{i+1}. [S-Skoru: {row['S_Skoru']:.2f}] {proje_isim}")
    print(f"   (İnsan Yazımı: {row['Insan_Puani']:.1f} | Proje Uyumu: {row['Baglam_Puani']:.1f})")
    print(f"   M-Skoru (Önceki): {row['M_Skoru']:.2f}")
    print("-" * 80)

# Kaydet
df_top20.to_pickle("IP3_Ciktisi_S_Skorlu_Projeler.pkl")
print("\nDosya kaydedildi: 'IP3_Ciktisi_S_Skorlu_Projeler.pkl'")

İP 3 Başladı: Samimiyet Skoru (S) Modülü...
-> 952 adet puanlanmış proje yüklendi.

Adım 2: Hile Tespiti (AI Detector) modeli yükleniyor...


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/624 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

Some weights of the model checkpoint at roberta-base-openai-detector were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 4216d318-2032-428f-8b98-61c759189cde)')' thrown while requesting HEAD https://huggingface.co/xlm-roberta-base/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].


-> Mektubun Yapay Zekâ Olma İhtimali: %99.95

Adım 3: Bağlamsal Benzerlik modeli yükleniyor (XLM-R)...

Samimiyet Skorları Hesaplanıyor (İlk 20 proje için)...

🏆 EN SAMİMİ EŞLEŞMELER (S-Skoru) 🏆
Referans Mektup AI Skoru: 0.9995 (Düşük olması iyidir)
--------------------------------------------------------------------------------
149. [S-Skoru: 59.73] 48828
   (İnsan Yazımı: 0.0 | Proje Uyumu: 99.5)
   M-Skoru (Önceki): 99.87
--------------------------------------------------------------------------------
348. [S-Skoru: 59.87] 47877
   (İnsan Yazımı: 0.0 | Proje Uyumu: 99.8)
   M-Skoru (Önceki): 99.86
--------------------------------------------------------------------------------
204. [S-Skoru: 59.80] 48664
   (İnsan Yazımı: 0.0 | Proje Uyumu: 99.6)
   M-Skoru (Önceki): 99.86
--------------------------------------------------------------------------------
826. [S-Skoru: 59.90] 33631
   (İnsan Yazımı: 0.0 | Proje Uyumu: 99.8)
   M-Skoru (Önceki): 99.85
----------------------------------

In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import pandas as pd
import numpy as np

print("İP 4 Başladı: Güvenilirlik Skoru (R) Modülü...")

# -------------------------------------------------------------------------
# ADIM 1: VERİLERİ HAZIRLA (İDDİA vs KANIT)
# -------------------------------------------------------------------------
try:
    # 1. İddialar (Claims): Adayın Beceri Listesi
    df_skills = pd.read_pickle("IP1_Ciktisi_Aday_Cikarilan_Beceriler.pkl")
    aday_iddialari = df_skills["cikarilan_beceri_adi"].tolist()
    print(f"-> {len(aday_iddialari)} adet 'İddia' (Beceri) yüklendi.")

    # 2. Kanıt (Evidence): CV'nin Tam Metni (Sertifikalar dahil)
    with open("IP1_Ciktisi_Aday_Tam_Metin.txt", "r", encoding="utf-8") as f:
        # Metni paragraflara bölüyoruz ki model boğulmasın
        cv_paragraphs = [p for p in f.read().split('\n') if len(p) > 20]
    print(f"-> {len(cv_paragraphs)} adet 'Kanıt Parçası' (CV satırları) yüklendi.")

except FileNotFoundError:
    print("HATA: İP 1 çıktıları bulunamadı.")
    raise

# -------------------------------------------------------------------------
# ADIM 2: NLI (Doğal Dil Çıkarımı) MODELİ
# -------------------------------------------------------------------------
print("\nAdım 2: NLI (Dijital Dedektif) modeli yükleniyor...")
# Bu model, (Kanıt, İddia) çiftini alır ve 'Entailment' (Doğrulama) skoru verir.
nli_model_name = "cross-encoder/nli-distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(nli_model_name)
model = AutoModelForSequenceClassification.from_pretrained(nli_model_name)

# M2 GPU Kullanımı
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model.to(device)
print(f"Model {device} cihazına yüklendi.")

def verify_claim(claim, evidence_list):
    """Bir iddianın (becerinin) kanıt listesi içinde geçip geçmediğini doğrular."""
    
    # İddiayı bir cümleye çevir: "I have the skill of Python."
    hypothesis = f"I have the skill of {claim}."
    
    # Kanıtlarla çiftler oluştur: [("Sertifika: Python 101", "Python biliyorum"), ...]
    pairs = [(evidence, hypothesis) for evidence in evidence_list]
    
    # Modele ver
    inputs = tokenizer(pairs, padding=True, truncation=True, return_tensors="pt", max_length=512).to(device)
    
    with torch.no_grad():
        logits = model(**inputs).logits
    
    # NLI Modeli Çıktısı: [Contradiction, Entailment, Neutral] (Sıra modele göre değişebilir)
    # cross-encoder/nli-distilroberta-base için etiketler:
    # 0: Contradiction (Yalanlıyor)
    # 1: Entailment (Doğruluyor) <-- Bizim aradığımız
    # 2: Neutral (Alakasız)
    
    # Her bir kanıt satırı için 'Entailment' (1. indeks) skorunu al
    entailment_scores = logits[:, 1].cpu().numpy()
    
    # En yüksek doğrulama skorunu bul (En az bir satır doğruluyorsa yeterli)
    max_score = entailment_scores.max()
    
    # Sigmoid benzeri bir dönüşümle 0-100 arası puana çevir (Logit -> Probability)
    # Basitçe: Pozitif logit güçlü doğrulama demektir.
    probability = 1 / (1 + np.exp(-max_score)) 
    
    return probability * 100

# -------------------------------------------------------------------------
# ADIM 3: TÜM BECERİLERİ DOĞRULA (GÜVENİLİRLİK SKORU)
# -------------------------------------------------------------------------
print("\nAdım 3: İddialar doğrulanıyor (Bu işlem biraz sürebilir)...")

dogrulama_sonuclari = []
toplam_skor = 0

for skill in aday_iddialari:
    # Her beceri için CV'deki kanıtları tara
    score = verify_claim(skill, cv_paragraphs)
    toplam_skor += score
    
    durum = "✅ Doğrulandı" if score > 50 else "❓ Kanıt Zayıf"
    dogrulama_sonuclari.append({"Beceri": skill, "R_Skoru": score, "Durum": durum})
    print(f"   Skill: {skill.ljust(30)} -> Skor: {score:.1f} ({durum})")

# GENEL GÜVENİLİRLİK SKORU (R)
# Tüm becerilerin ortalama doğrulanma oranı
final_r_score = toplam_skor / len(aday_iddialari)

print("-" * 60)
print(f"🏆 NİHAİ GÜVENİLİRLİK SKORU (R): {final_r_score:.2f} / 100")
print("-" * 60)

# Sonuçları Kaydet (İP 5 için sadece tek bir sayı lazım ama detayları da saklayalım)
df_r_details = pd.DataFrame(dogrulama_sonuclari)
df_r_details.to_pickle("IP4_Ciktisi_R_Detaylari.pkl")

# R Skorunu bir dosyaya tek sayı olarak kaydet (veya değişkende tut)
with open("IP4_Ciktisi_R_Skoru.txt", "w") as f:
    f.write(str(final_r_score))

print("Dosyalar kaydedildi.")

İP 4 Başladı: Güvenilirlik Skoru (R) Modülü...
-> 13 adet 'İddia' (Beceri) yüklendi.
-> 3 adet 'Kanıt Parçası' (CV satırları) yüklendi.

Adım 2: NLI (Dijital Dedektif) modeli yükleniyor...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/328M [00:00<?, ?B/s]

Model mps cihazına yüklendi.

Adım 3: İddialar doğrulanıyor (Bu işlem biraz sürebilir)...
   Skill: social medium                  -> Skor: 93.7 (✅ Doğrulandı)
   Skill: c                              -> Skor: 89.6 (✅ Doğrulandı)
   Skill: html                           -> Skor: 62.7 (✅ Doğrulandı)
   Skill: sql server                     -> Skor: 74.1 (✅ Doğrulandı)
   Skill: c++                            -> Skor: 76.9 (✅ Doğrulandı)
   Skill: microsoft powerpoint           -> Skor: 32.4 (❓ Kanıt Zayıf)
   Skill: microsoft excel                -> Skor: 24.1 (❓ Kanıt Zayıf)
   Skill: microsoft office               -> Skor: 25.8 (❓ Kanıt Zayıf)
   Skill: programming languages          -> Skor: 87.5 (✅ Doğrulandı)
   Skill: microsoft word                 -> Skor: 39.4 (❓ Kanıt Zayıf)
   Skill: network engineer               -> Skor: 93.9 (✅ Doğrulandı)
   Skill: zoom                           -> Skor: 45.6 (❓ Kanıt Zayıf)
   Skill: google drive                   -> Skor: 14.8 (❓ Kanıt Z

In [3]:
import pandas as pd
import numpy as np

print("İP 5 Başladı: Adil Sıralama (MOO) Modülü Entegrasyonu...")

# -------------------------------------------------------------------------
# ADIM 1: TÜM SKORLARI BİR ARAYA GETİR
# -------------------------------------------------------------------------
try:
    # 1. M ve S Skorlarını içeren tabloyu yükle (İP 3'ten gelen)
    df_final = pd.read_pickle("IP3_Ciktisi_S_Skorlu_Projeler.pkl")
    print(f"-> M ve S skorlu {len(df_final)} proje yüklendi.")
    
    # 2. R Skorunu yükle (İP 4'ten gelen tek değer)
    with open("IP4_Ciktisi_R_Skoru.txt", "r") as f:
        r_score = float(f.read())
    print(f"-> Adayın Güvenilirlik Skoru (R) yüklendi: {r_score:.2f}")
    
except FileNotFoundError:
    print("HATA: Önceki adımların çıktı dosyaları bulunamadı.")
    raise

# -------------------------------------------------------------------------
# ADIM 2: ÇOK AMAÇLI FAYDA FONKSİYONU (UTILITY FUNCTION)
# -------------------------------------------------------------------------
# Formül: Final_Score = (w1 * M) + (w2 * S) + (w3 * R)
# Ağırlıklar (Senaryo gereği):
# - Eşleşme (M): %50 (En önemlisi)
# - Samimiyet (S): %30 (Niyet önemli)
# - Güvenilirlik (R): %20 (CV doğruluğu)

w_m = 0.5
w_s = 0.3
w_r = 0.2

print(f"\nAğırlıklar: M=%{w_m*100}, S=%{w_s*100}, R=%{w_r*100}")

# R Skoru tek bir değer olduğu için tüm projelere aynı etkiyi yapar (Adayın özelliği)
# Ancak formülde yer alması, güvenilir adayların genel sıralamadaki yerini (diğer adaylara göre) belirler.
# Bizim prototipimizde tek aday olduğu için sıralamayı değiştirmez ama skoru etkiler.

def calculate_moo_score(row):
    m = row['M_Skoru']
    s = row['S_Skoru']
    r = r_score
    
    final_score = (m * w_m) + (s * w_s) + (r * w_r)
    return final_score

df_final['Final_Skor'] = df_final.apply(calculate_moo_score, axis=1)

# -------------------------------------------------------------------------
# ADIM 3: ADİLLİK (FAIRNESS) DÜZENLEMESİ (SİMÜLASYON)
# -------------------------------------------------------------------------
# CPFair mantığı:
# Popüler olmayan veya dezavantajlı projelerin görünürlüğünü artırmak.
# Simülasyon: "Kültür" veya "Sanat" ile ilgili projelere "Görünürlük Bonusu" verelim.

def apply_fairness_boost(row):
    bonus = 0
    # Proje açıklamasında 'culture' veya 'art' geçiyorsa ve puanı sınırda ise destekle
    text = row['tam_aciklama'].lower()
    if 'culture' in text or 'art' in text:
        bonus = 5.0 # +5 Puan Adillik Bonusu
        
    return row['Final_Skor'] + bonus

df_final['Adil_Skor'] = df_final.apply(apply_fairness_boost, axis=1)

# -------------------------------------------------------------------------
# ADIM 4: SONUÇLARI SIRALA VE RAPORLA
# -------------------------------------------------------------------------
df_final = df_final.sort_values(by='Adil_Skor', ascending=False)

print("\n🚀 NİHAİ ADİL SIRALAMA SONUÇLARI (TOP 5) 🚀")
print("-" * 100)
print(f"{'PROJE ADI':<50} | {'M-PUAN':<8} | {'S-PUAN':<8} | {'R-PUAN':<8} | {'FİNAL':<8}")
print("-" * 100)

for i, row in df_final.head(5).iterrows():
    proje_adi = row['link'].split('/')[-1].replace('_en', '').replace('-', ' ').upper()[:45]
    print(f"{proje_adi:<50} | {row['M_Skoru']:.1f}     | {row['S_Skoru']:.1f}     | {r_score:.1f}     | {row['Adil_Skor']:.2f}")

# Sonuçları Kaydet
df_final.to_csv("IP5_Nihai_Adil_Siralama.csv", index=False)
print("\n✅ Proje başarıyla tamamlandı! Sonuçlar 'IP5_Nihai_Adil_Siralama.csv' dosyasına kaydedildi.")

İP 5 Başladı: Adil Sıralama (MOO) Modülü Entegrasyonu...
-> M ve S skorlu 20 proje yüklendi.
-> Adayın Güvenilirlik Skoru (R) yüklendi: 58.49

Ağırlıklar: M=%50.0, S=%30.0, R=%20.0

🚀 NİHAİ ADİL SIRALAMA SONUÇLARI (TOP 5) 🚀
----------------------------------------------------------------------------------------------------
PROJE ADI                                          | M-PUAN   | S-PUAN   | R-PUAN   | FİNAL   
----------------------------------------------------------------------------------------------------
33631                                              | 99.9     | 59.9     | 58.5     | 84.59
47830                                              | 99.8     | 59.9     | 58.5     | 84.59
47877                                              | 99.9     | 59.9     | 58.5     | 84.59
37127                                              | 99.8     | 59.9     | 58.5     | 84.59
26159                                              | 99.8     | 59.9     | 58.5     | 84.58

✅ Proje başarıyla 